In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install catboost optuna shap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 20.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [ ]:
# 1. Load Data
train = pd.read_csv('/content/drive/MyDrive/스트레스 테스트 11차/train.csv')
test = pd.read_csv('/content/drive/MyDrive/스트레스 테스트 11차/test.csv')
submission = pd.read_csv('/content/drive/MyDrive/스트레스 테스트 11차/sample_submission.csv')

# 2. Data Preparation
X = train.drop(columns=['ID', 'stress_score'])
y = train['stress_score']
X_test = test.drop(columns=['ID'])

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 3. Preprocessing Pipeline
numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Transform data prior to tuning for efficiency
X_preprocessed = preprocessor.fit_transform(X)
X_test_preprocessed = preprocessor.transform(X_test)

# 4. Hyperparameter Tuning with Optuna
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'tree_method': 'hist'
    }
    model = XGBRegressor(**params)
    score = cross_val_score(model, X_preprocessed, y, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    return -score.mean()

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'random_state': 42,
        'verbose': -1
    }
    model = LGBMRegressor(**params)
    score = cross_val_score(model, X_preprocessed, y, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    return -score.mean()

def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'random_state': 42,
        'verbose': 0
    }
    model = CatBoostRegressor(**params)
    score = cross_val_score(model, X_preprocessed, y, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    return -score.mean()

# Adjust n_trials for actual implementation time limits
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=20)

study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=20)

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=20)

# 5. Model Training & Blending
rf_model = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
xgb_model = XGBRegressor(**study_xgb.best_params)
lgbm_model = LGBMRegressor(**study_lgbm.best_params)
cat_model = CatBoostRegressor(**study_cat.best_params, verbose=0)

# Group models into a VotingRegressor to handle blending seamlessly
voting_model = VotingRegressor(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('cat', cat_model)
    ],
    weights=[0.05, 0.25, 0.20, 0.50]
)

# Train the ensemble on the full dataset
voting_model.fit(X_preprocessed, y)

# 6. Prediction & Submission
# Generate predictions for the *test* set, not the training set
test_predictions = voting_model.predict(X_test_preprocessed)

# Bound checking: assuming stress score is typically between [0, 1]
test_predictions = np.clip(test_predictions, 0, 1)

submission['stress_score'] = test_predictions
submission.to_csv('/content/drive/MyDrive/스트레스 테스트 11차/sample_submission.csv', index=False, float_format='%.2f')

[I 2026-03-16 06:49:13,104] A new study created in memory with name: no-name-f9905182-a459-497f-8210-82bf45160dc3
[I 2026-03-16 06:49:25,251] Trial 0 finished with value: 0.2151257933607201 and parameters: {'n_estimators': 862, 'max_depth': 4, 'learning_rate': 0.03338410224208901, 'subsample': 0.6885642002479766, 'colsample_bytree': 0.7381332336707236}. Best is trial 0 with value: 0.2151257933607201.
[I 2026-03-16 06:49:29,055] Trial 1 finished with value: 0.22457198831458888 and parameters: {'n_estimators': 1333, 'max_depth': 3, 'learning_rate': 0.033160999097258136, 'subsample': 0.9897171666952521, 'colsample_bytree': 0.8591240056054559}. Best is trial 0 with value: 0.2151257933607201.
[I 2026-03-16 06:49:35,644] Trial 2 finished with value: 0.18965660553721087 and parameters: {'n_estimators': 824, 'max_depth': 6, 'learning_rate': 0.050162879929152415, 'subsample': 0.8432201333053094, 'colsample_bytree': 0.9949423889239846}. Best is trial 2 with value: 0.18965660553721087.
[I 2026-03

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000411 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1474
[LightGBM] [Info] Number of data points in the train set: 3000, number of used features: 29
[LightGBM] [Info] Start training from score 0.482130


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.model_selection import cross_val_score

# 앙상블 모델의 교차 검증 수행 (MAE 계산)
cv_scores = cross_val_score(voting_model, X_preprocessed, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)

# 결과 출력 (음수로 나오므로 -를 붙여 양수로 변환)
mae_estimate = -cv_scores.mean()
print(f"Ensemble Model Estimated MAE: {mae_estimate:.4f}")

Ensemble Model Estimated MAE: 0.1690


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# 데이터를 학습용과 검증용으로 분리 (8:2 비율)
X_train, X_val, y_train, y_val = train_test_split(X_preprocessed, y, test_size=0.2, random_state=42)

# 분리된 학습 데이터로 모델 재학습
voting_model.fit(X_train, y_train)

# 검증 데이터로 예측 및 MAE 계산
val_predictions = voting_model.predict(X_val)
val_mae = mean_absolute_error(y_val, val_predictions)

print(f"Validation MAE: {val_mae:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000278 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1466
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 29
[LightGBM] [Info] Start training from score 0.481912
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Validation MAE: 0.1597


In [14]:
submission = pd.read_csv('/content/drive/MyDrive/스트레스 테스트 11차/sample_submission.csv')

In [16]:
submission['stress_score'] = test_predictions
submission.head()

,ID,stress_score
0,TEST_0000,0.548257
1,TEST_0001,0.926550
2,TEST_0002,0.268165
3,TEST_0003,0.447720
4,TEST_0004,0.537692


In [17]:
submission.to_csv('/content/drive/MyDrive/스트레스 테스트 11차/sample_submission.csv', index=False, float_format='%.2f')